In [0]:
%pip install --upgrade databricks-sdk psycopg2-binary
dbutils.library.restartPython()

# Orphan Byte Sweeper

Scans all three UC Volumes (`session_audio`, `screenshots`, `documents`) for files
that do NOT have a corresponding row in `app.uploads`. These "orphan bytes" can occur
when the App writes a file to a Volume but the subsequent Lakebase INSERT fails.

**Behavior:** Report-only. Logs orphan file paths, sizes, and totals. Does NOT delete.

**Parameters:**
- `catalog` — UC catalog name
- `schema` — UC schema name
- `lakebase_project_id` — Lakebase project ID for endpoint discovery
- `lakebase_database_id` — Lakebase database resource ID

**Schedule:** Weekly (Sunday 2:00 AM UTC) via `orphan_byte_sweeper` job.

In [0]:
dbutils.widgets.text("catalog", "", "UC Catalog")
dbutils.widgets.text("schema", "", "UC Schema")
dbutils.widgets.text("lakebase_project_id", "", "Lakebase Project ID")
dbutils.widgets.text("lakebase_database_id", "", "Lakebase Database ID")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
lakebase_project_id = dbutils.widgets.get("lakebase_project_id")
lakebase_database_id = dbutils.widgets.get("lakebase_database_id")

assert catalog, "catalog is required"
assert schema, "schema is required"
assert lakebase_project_id, "lakebase_project_id is required"
assert lakebase_database_id, "lakebase_database_id is required"

VOLUMES = ["session_audio", "screenshots", "documents"]
print(f"  Catalog: {catalog}")
print(f"  Schema:  {schema}")
print(f"  Volumes: {VOLUMES}")
print(f"  Lakebase project: {lakebase_project_id}")

In [0]:
from datetime import datetime, timezone

def list_volume_files(volume_path: str) -> list[dict]:
    """Recursively list all files in a UC Volume path."""
    files = []
    try:
        items = dbutils.fs.ls(volume_path)
    except Exception as e:
        print(f"  Warning: Could not list {volume_path}: {e}")
        return files

    for item in items:
        if item.isDir():
            files.extend(list_volume_files(item.path))
        else:
            files.append({
                "path": item.path,
                "size_bytes": item.size,
                "name": item.name,
            })
    return files

# Scan all three volumes
all_volume_files: dict[str, list[dict]] = {}
total_files = 0
total_bytes = 0

print(f"Scanning volumes at {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}...")
print()

for vol in VOLUMES:
    volume_path = f"/Volumes/{catalog}/{schema}/{vol}"
    print(f"  Scanning: {volume_path}")
    files = list_volume_files(volume_path)
    all_volume_files[vol] = files
    vol_bytes = sum(f["size_bytes"] for f in files)
    total_files += len(files)
    total_bytes += vol_bytes
    print(f"    Found {len(files)} files ({vol_bytes / 1024 / 1024:.2f} MB)")

print()
print(f"  Total: {total_files} files across {len(VOLUMES)} volumes ({total_bytes / 1024 / 1024:.2f} MB)")

In [0]:
import requests as req
import json as json_mod
from base64 import urlsafe_b64decode
from databricks.sdk import WorkspaceClient
import psycopg2

w = WorkspaceClient()

# Resolve Lakebase endpoint
branch_path = f"projects/{lakebase_project_id}/branches/production"
auth_headers = w.config.authenticate()
host_url = w.config.host

resp = req.get(
    f"{host_url}/api/2.0/postgres/{branch_path}/endpoints",
    headers=auth_headers
)
resp.raise_for_status()
endpoints_data = resp.json().get("endpoints", [])

if not endpoints_data:
    raise RuntimeError(f"No endpoint found for '{branch_path}'.")

ep = endpoints_data[0]
pg_host = ep["status"]["hosts"]["host"]
pg_port = 5432

# Generate credential
cred_resp = req.post(
    f"{host_url}/api/2.0/postgres/credentials",
    headers=auth_headers,
    json={"endpoint": ep["name"]}
)
cred_resp.raise_for_status()
cred = cred_resp.json()
pg_token = cred["token"]

# Extract PG username from JWT
parts = pg_token.split(".")
payload_b64 = parts[1] + "=" * (4 - len(parts[1]) % 4)
payload = json_mod.loads(urlsafe_b64decode(payload_b64))
pg_user = payload.get("sub", "")

print(f"  Connected to Lakebase: {pg_host}")
print(f"  PG user: {pg_user}")

# Query all tracked upload paths
conn = psycopg2.connect(
    host=pg_host, port=pg_port, dbname="databricks_postgres",
    user=pg_user, password=pg_token, sslmode="require"
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("SELECT volume_path FROM app.uploads WHERE revoked_at IS NULL")
tracked_paths = {row[0] for row in cur.fetchall()}

cur.close()
conn.close()

print(f"  Tracked uploads in app.uploads: {len(tracked_paths)}")

In [0]:
# Compare volume files against tracked uploads
# Note: dbutils.fs.ls returns paths like "dbfs:/Volumes/..." but app.uploads
# stores paths like "/Volumes/...". Normalize for comparison.

def normalize_path(p: str) -> str:
    """Strip dbfs: prefix if present."""
    if p.startswith("dbfs:"):
        return p[5:]
    return p

orphans: list[dict] = []
orphan_bytes = 0

for vol, files in all_volume_files.items():
    for f in files:
        normalized = normalize_path(f["path"])
        if normalized not in tracked_paths:
            orphans.append({
                "volume": vol,
                "path": normalized,
                "size_bytes": f["size_bytes"],
            })
            orphan_bytes += f["size_bytes"]

print(f"Orphan analysis complete:")
print(f"  Total files on volumes:    {total_files}")
print(f"  Tracked in app.uploads:    {len(tracked_paths)}")
print(f"  Orphan files found:        {len(orphans)}")
print(f"  Orphan bytes:              {orphan_bytes / 1024 / 1024:.2f} MB")
print()

if not orphans:
    print("  ✓ No orphan files detected. All volume files have matching upload records.")

In [0]:
from datetime import datetime, timezone

if orphans:
    print("=" * 70)
    print("ORPHAN FILE REPORT")
    print(f"Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
    print("=" * 70)
    print()

    # Group by volume
    for vol in VOLUMES:
        vol_orphans = [o for o in orphans if o["volume"] == vol]
        if not vol_orphans:
            continue
        vol_bytes = sum(o["size_bytes"] for o in vol_orphans)
        print(f"── {vol} ({len(vol_orphans)} orphans, {vol_bytes / 1024 / 1024:.2f} MB) ──")
        for o in sorted(vol_orphans, key=lambda x: x["path"]):
            size_kb = o["size_bytes"] / 1024
            print(f"  {o['path']}  ({size_kb:.1f} KB)")
        print()

    print("=" * 70)
    print(f"ACTION REQUIRED: {len(orphans)} orphan file(s) detected.")
    print("These files exist on UC Volumes but have no matching app.uploads row.")
    print("Possible causes:")
    print("  • App crashed between Volume write and Lakebase INSERT")
    print("  • Legacy files from pre-rename layout (expected in dev)")
    print("  • Manual test uploads without going through the API")
    print()
    print("To clean up, manually verify and delete via:")
    print("  dbutils.fs.rm('<path>')")
    print("=" * 70)
else:
    print("✓ Volume integrity check passed — no orphans.")

# Set task value for downstream consumption (e.g., alerting)
dbutils.jobs.taskValues.set(key="orphan_count", value=len(orphans))
dbutils.jobs.taskValues.set(key="orphan_bytes", value=orphan_bytes)